<a href="https://colab.research.google.com/github/srinivas340/ml_practice/blob/main/greadent_discent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [372]:
import numpy as np
import torch

training data

In [373]:
#Input(temp,rainfall, humidity)
inputs=np.array([[73,67,43],
                 [91,88,64],
                 [87,134,58],
                 [102,43,37],
                 [69,96,70]],dtype='float32')

In [374]:
#Targets(apples,oranges)
targets = np.array([[56,70],
                    [81,101],
                    [119,133],
                    [22,37],
                    [103,119]],dtype='float32')


In [375]:
#convert inputs and outputs to tensors
inputs = torch.from_numpy(inputs)
targets = torch.from_numpy(targets)
print(inputs)
print(targets)

tensor([[ 73.,  67.,  43.],
        [ 91.,  88.,  64.],
        [ 87., 134.,  58.],
        [102.,  43.,  37.],
        [ 69.,  96.,  70.]])
tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])


In [376]:
#weights and biases
w=torch.randn(2,3,requires_grad=True)
b=torch.randn(2,requires_grad=True)
print(w)
print(b)

tensor([[ 0.1157,  2.0453,  0.1072],
        [ 0.6952, -0.7216,  0.8032]], requires_grad=True)
tensor([ 0.9553, -0.0576], requires_grad=True)


In [377]:
def model(x):
  return x @ w.t() + b

@ represent matrix multiplication in pytorch

In [378]:
#generate predictions
preds = model(inputs)
print(preds)

tensor([[151.0439,  36.8805],
        [198.3282,  51.1076],
        [291.3046,  10.3124],
        [104.6700,  69.5402],
        [212.7878,  34.8602]], grad_fn=<AddBackward0>)


In [379]:
#compare with targets
print(targets)

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])


In [380]:
diff=preds-targets
diff_sqr=diff*diff
torch.sum(diff_sqr)/diff.numel()

tensor(9815.2598, grad_fn=<DivBackward0>)

In [381]:
#mse loss
def mse(t1,t2):
  diff=t1-t2
  return torch.sum(diff*diff)/diff.numel()

In [382]:
#compute loss
loss =mse(preds,targets)
print(loss)

tensor(9815.2598, grad_fn=<DivBackward0>)


Compute gradients, we can automatically comute the gradient or derivative of the loss w.r.t to the weights and biases, because they have (requires_grad  set to True)

In [383]:
# Compute gradients
loss.backward()

the gradients are sorted in the (.grad) property of the respective tensors. note that the derivative of the loss w.r.t the weights matrix is itself a matrix is itself a matrix,with the same dimentions

In [384]:
#gradients for weights
print(w)
print(w.grad)

tensor([[ 0.1157,  2.0453,  0.1072],
        [ 0.6952, -0.7216,  0.8032]], requires_grad=True)
tensor([[ 9722.6553, 10775.2168,  6466.6997],
        [-4023.6609, -5945.5742, -3283.7871]])


In [385]:
print(b)
print(b.grad)

tensor([ 0.9553, -0.0576], requires_grad=True)
tensor([115.4269, -51.4598])


In [386]:
w.grad.zero_()
b.grad.zero_()
print(w.grad)
print(b.grad)


tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([0., 0.])


adjust weights and biases using greadient descent

1.generate predictions

2.caliculate the loss

3.compute gradients w.r.t the weights and bias

4.adjust the weights by subtracting a small quantity proportional to the gradient

5.resent the gradients to the zero


In [387]:
#generate predictions
preds =model(inputs)
print(preds)

tensor([[151.0439,  36.8805],
        [198.3282,  51.1076],
        [291.3046,  10.3124],
        [104.6700,  69.5402],
        [212.7878,  34.8602]], grad_fn=<AddBackward0>)


In [388]:
#calculate the loss
loss = mse(preds,targets)
print(loss)

tensor(9815.2598, grad_fn=<DivBackward0>)


In [389]:
#compute gradients
loss.backward()
print(w.grad)
print(b.grad)

tensor([[ 9722.6553, 10775.2168,  6466.6997],
        [-4023.6609, -5945.5742, -3283.7871]])
tensor([115.4269, -51.4598])


finally, we update the weights and biases using the gradients computed above.

In [390]:
#Adjust weights and reset gradients
with torch.no_grad():
  w-=w.grad*1e-5
  b-=b.grad*1e-5
  w.grad.zero_()
  b.grad.zero_()

In [391]:
print(w)
print(b)

tensor([[ 0.0185,  1.9375,  0.0425],
        [ 0.7354, -0.6622,  0.8361]], requires_grad=True)
tensor([ 0.9541, -0.0571], requires_grad=True)


wite the new weights and biases, the model should have lower loss

In [392]:
#calculate loss
preds=model(inputs)
loss=mse(preds,targets)
print(loss)

tensor(6948.1133, grad_fn=<DivBackward0>)


we have already achived a significant reduction in the loss, simply by adjusting the weights and biases slightly using greadient descent

**Train for multiple epochs**

To reduce the loss further, we can repeat the process of adjusting the weights and biases slightly using gradient descent

In [393]:
#train for 100 epochs
for i in range(2000):
  preds=model(inputs)
  loss =mse(preds,targets)
  loss.backward()
  with torch.no_grad():
    w-=w.grad*1e-5
    b-=b.grad*1e-5
    w.grad.zero_()
    b.grad.zero_()

lets verify loss it is lower:

In [394]:
#Calculate loss
preds=model(inputs)
loss=mse(preds,targets)
print(loss)


tensor(1.8446, grad_fn=<DivBackward0>)


In [395]:
#predictions
preds

tensor([[ 57.4582,  70.2695],
        [ 80.9869, 100.9173],
        [120.9997, 132.4598],
        [ 21.7148,  36.8768],
        [ 99.6164, 119.6357]], grad_fn=<AddBackward0>)

In [396]:
#targets
targets

tensor([[ 56.,  70.],
        [ 81., 101.],
        [119., 133.],
        [ 22.,  37.],
        [103., 119.]])